In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import json
import joblib

from scipy.stats import randint
from functools import partial

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import make_scorer, ndcg_score
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures

import sys
from pathlib import Path

In [2]:
pd.set_option('display.max_columns', None)

ROOT = Path.cwd()
while not (ROOT / ".git").exists() and ROOT != ROOT.parent:
     ROOT = ROOT.parent
src_path = str(ROOT/"src")
if src_path not in sys.path:
    sys.path.insert(0,src_path)

from config import START_YEAR, END_YEAR, HOLDOUT_START
import utils
from final_model import ClassifiedRegressor

In [3]:
df = utils.player_stat_dataframe(START_YEAR,END_YEAR)

train = df[df.Year < HOLDOUT_START]
test = df[df.Year >= HOLDOUT_START]

req = ['Year', 'GP%', 'MP']
f = ['PTS','W%','DRB','AST','BLK','STL','FT','3P','eFG%']
features = list(set(req + f))

target = 'MVPVS'

In [4]:
X_tr, y_tr = train[features], train[target]

cv = list(utils.forward_chaining_splits(X_tr.Year, min_train_years=10))

def ndcg_at_1(y_true, y_pred):
    return ndcg_score(np.asarray(y_true).reshape(1, -1),
                      np.asarray(y_pred).reshape(1, -1), k=1)

ndcg1_scorer = make_scorer(ndcg_at_1)

base = ClassifiedRegressor(
    classifier=make_pipeline(
        PolynomialFeatures(degree=2), 
        StandardScaler(),
        LogisticRegression(solver='saga', max_iter=1000, random_state=97, tol=1e-3)),
    regressor=HistGradientBoostingRegressor(random_state=29),
)

search = RandomizedSearchCV(
    base,
    param_distributions={
        'classifier__logisticregression__C': np.logspace(-2, 1, 5),
        'classifier__logisticregression__l1_ratio': [0, 0.25, 0.5, 0.75, 1],
        'threshold': [0.1, 0.3, 0.5, 0.7, 0.9],
        'regressor__loss': ['squared_error', 'poisson'],
        'regressor__learning_rate': [0.01, 0.03, 0.05],
        'regressor__max_iter': [100, 200, 300, 400, 500],
        'regressor__max_depth': [2, 3, 4, None],
        'regressor__max_leaf_nodes': [7, 15, 31],
        'regressor__min_samples_leaf': [1, 2, 3, 4, 5],
        'regressor__l2_regularization': [0.0, 0.5, 1.0],
    },
    n_iter=100, cv=cv, scoring=ndcg1_scorer, n_jobs=-1, random_state=29,
)
search.fit(X_tr, y_tr)
print(f'Best parameters:')
print(search.best_params_)

model = search.best_estimator_


Best parameters:
{'threshold': 0.1, 'regressor__min_samples_leaf': 1, 'regressor__max_leaf_nodes': 15, 'regressor__max_iter': 200, 'regressor__max_depth': None, 'regressor__loss': 'squared_error', 'regressor__learning_rate': 0.01, 'regressor__l2_regularization': 0.0, 'classifier__logisticregression__l1_ratio': 0.5, 'classifier__logisticregression__C': np.float64(0.01)}


In [9]:
X_tr, y_tr = train[features], train[target]

tot = 0
fin = pd.DataFrame()
cv = list(utils.forward_chaining_splits(X_tr.Year, min_train_years=10))
for chain in cv:
    model = ClassifiedRegressor(
        classifier=make_pipeline(
        PolynomialFeatures(degree=2), 
        StandardScaler(),
        LogisticRegression(solver='saga', max_iter=1000, random_state=97, tol=1e-3)),
    regressor=HistGradientBoostingRegressor(random_state=29),
    )
    model = search.best_estimator_
    model.fit(X_tr.iloc[chain[0]], y_tr.iloc[chain[0]])
    res = train.iloc[chain[1]]
    res['Output'] = model.predict(X_tr.iloc[chain[1]])
    pred_winner = res.sort_values('Output', ascending=False).Player.iloc[:3].to_numpy()
    actual_winner = res.sort_values('MVPVS', ascending=False).Player.iloc[0]
    tot += (pred_winner[0] == actual_winner)
    fin = pd.concat([fin,pd.DataFrame({'Year':[res.Year.iloc[0]], 
                                       'Actual':[actual_winner], 
                                       'Predicted': [pred_winner], 
                                       'Top3': [(pred_winner == actual_winner).any()],
                                       'Match': [(pred_winner[0] == actual_winner)]})]
                    )

print(f'{tot}/{len(cv)}')
fin

28/35


,Year,Actual,Predicted,Top3,Match
0,1990,Michael Jordan,"[Michael Jordan, Magic Johnson, Karl Malone]",True,True
0,1991,Michael Jordan,"[Michael Jordan, Clyde Drexler, Chris Mullin]",True,True
0,1992,Charles Barkley,"[Charles Barkley, Patrick Ewing, Michael Jordan]",True,True
0,1993,Hakeem Olajuwon,"[Hakeem Olajuwon, David Robinson, Shaquille O'...",True,True
0,1994,David Robinson,"[David Robinson, Karl Malone, Hakeem Olajuwon]",True,True
0,1995,Michael Jordan,"[Michael Jordan, Karl Malone, David Robinson]",True,True
0,1996,Karl Malone,"[Michael Jordan, Karl Malone, Grant Hill]",True,False
0,1997,Michael Jordan,"[Michael Jordan, Shaquille O'Neal, Karl Malone]",True,True
0,1998,Karl Malone,"[Karl Malone, Allen Iverson, Shaquille O'Neal]",True,True
0,1999,Shaquille O'Neal,"[Shaquille O'Neal, Karl Malone, Kobe Bryant]",True,True


In [7]:
X_te, y_te = test[features], test[target]

tot = 0
fin = pd.DataFrame()
cv = list(utils.forward_chaining_splits(df.Year, min_train_years=10))[-len(test.Year.unique()):]
for chain in cv:
    model = search.best_estimator_
    model = ClassifiedRegressor(
            classifier=make_pipeline(
            PolynomialFeatures(degree=2), 
            StandardScaler(),
            LogisticRegression(solver='saga', max_iter=1000, random_state=97, tol=1e-3)),
        regressor=HistGradientBoostingRegressor(random_state=29),
        )
    model = search.best_estimator_
    model.fit(df[features].iloc[chain[0]], df[target].iloc[chain[0]])
    res = test.loc[chain[1]]
    res['Output'] = model.predict(df[features].iloc[chain[1]])
    pred_winner = res.sort_values('Output', ascending=False).Player.iloc[:3].to_numpy()
    actual_winner = res.sort_values('MVPVS', ascending=False).Player.iloc[0]
    tot += (pred_winner[0] == actual_winner)
    fin = pd.concat([fin,pd.DataFrame({'Year':[res.Year.iloc[0]], 
                                        'Actual':[actual_winner], 
                                        'Predicted': [pred_winner], 
                                        'Top3': [(pred_winner == actual_winner).any()],
                                        'Match': [(pred_winner[0] == actual_winner)]})]
    )

print(f'{tot}/{len(cv)}')
fin

1/1


,Year,Actual,Predicted,Top3,Match
0,2025,Shai Gilgeous-Alexander,"[Shai Gilgeous-Alexander, Nikola Jokić, Cade C...",True,True
